In [2]:
import os

from word_search_engine import Document, TextProcessor


class LongestWordFinder:
    def __init__(self):
        self._processor = TextProcessor()   # no stop words: every word counts

    def in_document(self, doc):
        """Return (length, {word: first_line_no}) for the longest word(s) in one file."""
        best_len = 0
        best = {}
        for line_no, _, text in doc.scan():
            for word in self._processor.tokenize(text):
                if len(word) > best_len:
                    best_len = len(word)
                    best = {word: line_no}
                elif len(word) == best_len and word not in best:
                    best[word] = line_no
        return best_len, best

    def in_documents(self, docs):
        """Return (overall_length, [(file_name, word, line_no), ...]) across many files."""
        per_file = {d.name: self.in_document(d) for d in docs}
        overall = max((length for length, _ in per_file.values()), default=0)
        matches = []
        if overall > 0:
            for name, (length, words) in per_file.items():
                if length == overall:
                    matches.extend((name, w, ln) for w, ln in words.items())
        return overall, matches

    def top_n(self, doc, n=5):
        """Return the n longest unique words in a file as [(word, first_line_no), ...]."""
        unique = {}
        for line_no, _, text in doc.scan():
            for word in self._processor.tokenize(text):
                unique.setdefault(word, line_no)
        ranked = sorted(unique.items(), key=lambda item: (-len(item[0]), item[0]))
        return ranked[:n]


# --------------------------------------------------------------------------
# Console menu
# --------------------------------------------------------------------------
def load_folder(folder):
    names = sorted(f for f in os.listdir(folder) if f.lower().endswith(".txt"))
    return [Document(os.path.join(folder, n)) for n in names]


def main():
    finder = LongestWordFinder()
    menu = """
=== Longest Word Finder ===
1. Longest word in a file
2. Longest word across a folder of .txt files
3. Top 5 longest words in a file
4. Exit
"""
    while True:
        print(menu)
        choice = input("Choose: ").strip()
        try:
            if choice == "1":
                doc = Document(input("File path: ").strip())
                length, words = finder.in_document(doc)
                if not words:
                    print("The file has no words.")
                for word, line_no in words.items():
                    print(f"Longest word: '{word}' ({length} letters), first seen on line {line_no}")
            elif choice == "2":
                docs = load_folder(input("Folder path: ").strip())
                length, matches = finder.in_documents(docs)
                if not matches:
                    print("No words found.")
                for name, word, line_no in matches:
                    print(f"'{word}' ({length} letters) in {name}, line {line_no}")
            elif choice == "3":
                doc = Document(input("File path: ").strip())
                for rank, (word, line_no) in enumerate(finder.top_n(doc), start=1):
                    print(f"{rank}. {word} ({len(word)} letters, line {line_no})")
            elif choice == "4":
                print("Goodbye!")
                break
            else:
                print("Invalid choice.")
        except (FileNotFoundError, NotADirectoryError) as e:
            print("Error:", e)


if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'word_search_engine'